In [ ]:
import pandas as pd
import os

In [ ]:
os.chdir('..')

In [ ]:
os.getcwd()

In [ ]:
data = pd.read_csv(r'data/raw/data.csv')
train = pd.read_csv(r'data/raw/train.csv')
test = pd.read_csv(r'data/raw/test.csv')

In [ ]:
data.head()

In [ ]:
data = data[['text','label']]

In [ ]:
data.tail(101)

In [ ]:
' '.join(data['text'][3].split())

In [ ]:
' '.join(data['text'][44867].split())

In [ ]:
import pandas as pd
import re
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
import nltk
from nltk.data import find

try:
    find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")
from nltk.corpus import stopwords

class BasicFeatureGenerator(BaseEstimator, TransformerMixin):
    def __init__(self):
        # Load resources once during initialization for speed
        self.stop_words = set(stopwords.words("english"))
        self.punctuation_pattern = re.compile(r"[.,!?;:\'\"()\[\]{}\-\—…]")
        self.number_pattern = re.compile(r"\d+")
        self.symbol_pattern = re.compile(r"[@#$%^&*+=|\\/<>~_]")

    def get_cleaned_string(self, text):
        if not isinstance(text, str): return ""
        text = text.lower()
        # Removes special characters but keeps alphanumeric and spaces
        text = re.sub(r"[^\w\s]", "", text)
        return " ".join(text.split())

    def get_punctuation_count(self, text):
        return len(self.punctuation_pattern.findall(text))

    def get_avg_word_length(self, text):
        words = text.split()
        if not words: return 0.0
        return sum(len(word) for word in words) / len(words)

    def get_capital_words_count(self, text):
        words = text.split()
        return sum(1 for word in words if word.isupper() and any(c.isalpha() for c in word))

    def get_stopword_count(self, text):
        words = text.lower().split()
        return sum(1 for word in words if word in self.stop_words)

    def fit(self, X, y=None):
        return self

    def transform(self, data: pd.DataFrame) -> pd.DataFrame:
        df = data.copy()
        
        # Safety Check
        if "text" not in df.columns:
            raise KeyError("The input DataFrame must contain a 'text' column.")

        # 1. Generate Cleaned Text (Required for the Embedding Generator stage)
        df["cleaned_text"] = [self.get_cleaned_string(t) for t in df["text"]]

        # 2. Vectorized Pandas Operations for Basic Stats
        df["text_character_count"] = df["text"].str.len()
        df["text_word_count"] = df["text"].str.split().str.len()
        
        # 3. Optimized Feature Extraction using List Comprehensions
        text_list = df["text"].tolist()
        
        df["text_stopword_count"] = [self.get_stopword_count(t) for t in text_list]
        df["text_unique_word_count"] = [len(set(t.split())) for t in text_list]
        df["text_punctuation_count"] = [self.get_punctuation_count(t) for t in text_list]
        df["text_avg_word_length"] = [self.get_avg_word_length(t) for t in text_list]
        df["text_capital_words_count"] = [self.get_capital_words_count(t) for t in text_list]
        
        # Regex-based features
        df["text_number_count"] = [len(self.number_pattern.findall(t)) for t in text_list]
        df["text_symbol_count"] = [len(self.symbol_pattern.findall(t)) for t in text_list]

        # 4. Final Cleanup: Drop the raw 'text' column. 
        # We keep 'cleaned_text' because the next step in your Pipeline needs it.
        df.drop(columns=["text"], inplace=True)

        return df

    def fit(self, X, y=None):
        return self

In [ ]:
bfg = BasicFeatureGenerator()
bf_data= bfg.transform(train[['text']])

In [ ]:
bf_data['label'] = train.label

In [ ]:
bf_data.columns

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# List of columns to plot
cols = [
    'text_character_count', 'text_word_count', 'text_stopword_count', 
    'text_unique_word_count', 'text_punctuation_count', 'text_avg_word_length',
    'text_capital_words_count', 'text_number_count', 'text_symbol_count'
]

# Create a 3x3 grid of plots
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(15, 12))
axes = axes.flatten() # Flatten the 2D array of axes for easy looping

for i, col in enumerate(cols):
    sns.histplot(
        data=bf_data, 
        x=col, 
        hue='label', 
        kde=True,        # Adds a smooth distribution curve
        element='step',  # Makes overlapping bars easier to see
        ax=axes[i]
    )
    axes[i].set_title(f'Dist of {col}')
    axes[i].set_xlabel('')

plt.tight_layout()
plt.show()


In [ ]:
!pip install wordcloud

In [ ]:
import sys
!{sys.executable} -m pip install wordcloud


In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# The function now looks specifically for 'cleaned_text'
def plot_wordcloud(label_value):
    # Combine all messages for the given label
    subset = bf_data[bf_data['label'] == label_value]['cleaned_text'].astype(str)
    combined_text = " ".join(subset)
    
    # Generate the cloud
    wc = WordCloud(width=800, height=400, background_color='white', 
                   colormap='viridis', max_words=150).generate(combined_text)
    return wc

# Plotting side-by-side
plt.figure(figsize=(16, 8))

# Category 0
plt.subplot(1, 2, 1)
plt.imshow(plot_wordcloud(0), interpolation='bilinear')
plt.title('Word Cloud: Label 0 (Cleaned)', fontsize=15)
plt.axis('off')

# Category 1
plt.subplot(1, 2, 2)
plt.imshow(plot_wordcloud(1), interpolation='bilinear')
plt.title('Word Cloud: Label 1 (Cleaned)', fontsize=15)
plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
!{sys.executable} -m pip install sentence_transformers


In [ ]:
!{sys.executable} -m pip install --upgrade typing-extensions



In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np
import pandas as pd

class EmbeddingFeaturesGenerator(BaseEstimator, TransformerMixin):
    def __init__(self, model_path, model_name="all-MiniLM-L6-v2"):
        self.model_path = model_path
        self.model_name = model_name
        self.model = None

    def fit(self, X, y=None):
        # Loading the model here ensures it only loads when needed
        self.model = SentenceTransformer(self.model_name, cache_folder=self.model_path)
        return self

    def transform(self, X):
        # 1. Validation: Ensure we have the cleaned text from the previous stage
        if "cleaned_text" not in X.columns:
            raise KeyError("EmbeddingFeaturesGenerator requires 'cleaned_text' column.")

        # 2. Semantic Embedding
        # We transform the text into a 384-dimensional vector (for MiniLM)
        # This captures the 'essence' of the writing style.
        text_embeddings = self.model.encode(
            X["cleaned_text"].tolist(), 
            show_progress_bar=False,
            convert_to_numpy=True
        )

        # 3. Numeric Feature Extraction
        # We drop all string columns. 
        # Only the numerical stats from BasicFeatureGenerator should remain.
        COLS_TO_REMOVE = [
            "id", "topic", "text", "label", 
            "prompt_name", "source", "RDizzl3_seven",  
            "cleaned_text", "cleaned_topics", "answer"
        ]
        
        # Extract numeric stats (character counts, word counts, etc.)
        numeric_stats = X.drop(columns=COLS_TO_REMOVE, errors='ignore').select_dtypes(include=[np.number]).to_numpy()

        # 4. Concatenate: [Statistical Features] + [Semantic Embeddings]
        # This gives XGBoost both the 'math' of the text and the 'meaning' of the text.
        final_features = np.column_stack(
            (
                numeric_stats,
                text_embeddings,
            )
        )

        return final_features

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline

from src.utils.logger import logger
from src.utils.common import (
    read_csv_file,
    save_numpy,
    assert_file_exists,
    log_file_size,
    save_object,
)
from src.utils.exception import AITextException
from src.entity.config_entity import DataTransformationConfig
from src.entity.artifact_entity import DataTransformationArtifact
from src.feature_generation.basic_features import BasicFeatureGenerator
from src.feature_generation.transformer_embedding import EmbeddingFeaturesGenerator


class DataTransformation:
    def __init__(self, cfg: DataTransformationConfig) -> None:
        self.cfg = cfg

    def split_data(self, df: pd.DataFrame):
        X = df.drop(columns=[self.cfg.target_column_name])
        y = df[self.cfg.target_column_name]

        return X, y

    def _identity_func(self, X):
        return X

    def initiate_data_transformation(self) -> DataTransformationArtifact:
        try:
            logger.info("Starting data transformation process")

            # Load validated CSVs
            df_train = read_csv_file(self.cfg.validated_data_train_path)
            # df_train = df_train.sample(50)
            # Train/val split
            X, y = self.split_data(df_train)

            # Pipeline creation
            pipeline = Pipeline(
                [
                    ("basic", BasicFeatureGenerator()),
                    (
                        "embedding",
                        EmbeddingFeaturesGenerator(model_path=self.cfg.temp_model_dir),
                    ),
                    ("identity", FunctionTransformer(self._identity_func)),
                ]
            )

            X = pipeline.fit_transform(X)

            # Save train data
            save_numpy(np.column_stack((X, y)), self.cfg.transformed_train_data_path)
            assert_file_exists(self.cfg.transformed_train_data_path, "Train numpy")
            log_file_size(self.cfg.transformed_train_data_path, "Train numpy")

            # Save pipeline
            save_object(pipeline, self.cfg.data_transformation_object_path)
            assert_file_exists(
                self.cfg.data_transformation_object_path, "Transform pipeline"
            )
            log_file_size(self.cfg.data_transformation_object_path, "Pipeline")

            logger.info("Data transformation process completed successfully")

            return DataTransformationArtifact(
                data_transformation_object_path=self.cfg.data_transformation_object_path,
                transformed_data_dir=os.path.dirname(
                    self.cfg.transformed_train_data_path
                ),
            )

        except Exception as e:
            logger.error("Data Transformation process failed")
            raise AITextException(e)
